# Predicting election winners with XGBoost

Loads the long poll dataset (`polls_long_with_results.csv` from `build_dataset.ipynb`), engineers features, collapses to **one row per candidate per race**, and trains a gradient-boosted classifier for `won`.

**Honest framing up front:** for *calling the winner*, the polling leader is already right ~93% of the time, so a model barely beats that on accuracy. The model's value is **calibrated win probabilities** (useful for close races, ratings, betting markets) — judged by AUC / log-loss / Brier, not just accuracy.

**Validation:** split by year — train on 2018/2020/2022, test on 2024 — so we never leak the future.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (accuracy_score, roc_auc_score, log_loss,
                             brier_score_loss, confusion_matrix)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
print('xgboost', xgb.__version__)

xgboost 3.2.0


## 1. Load & filter

Keep only polls that matched a result (`has_result == 1`) and general-election cycles (even years 2018–2024).

In [2]:
d = pd.read_csv('polls_long_with_results.csv', low_memory=False)
d = d[d['has_result'] == 1].copy()

d['end_date']      = pd.to_datetime(d['end_date'], errors='coerce', format='mixed')
d['election_date'] = pd.to_datetime(d['election_date'], errors='coerce', format='mixed')
d['days_to_elec']  = (d['election_date'] - d['end_date']).dt.days

CYCLES = [2018, 2020, 2022, 2024]
d = d[d['year'].isin(CYCLES)].copy()

for c in ['pct','numeric_grade','pollscore','transparency_score','sample_size','days_to_elec']:
    d[c] = pd.to_numeric(d[c], errors='coerce')

print('poll rows:', len(d))
print('races:', d['race_id'].nunique())
print('by year (races):', d.groupby('year')['race_id'].nunique().to_dict())

poll rows: 16756
races: 367
by year (races): {2018: 109, 2020: 78, 2022: 102, 2024: 78}


## 2. Poll weighting

Not all polls are equal. We weight each poll by:
- **recency** — closer to election day counts more (`1 / (1 + days/30)`),
- **sample size** — `sqrt(n)` (precision scales with root-n),
- **pollster quality** — 538 `numeric_grade`.

Used to compute a weighted polling average per candidate.

In [3]:
d['w'] = (1.0 / (1.0 + d['days_to_elec'].clip(lower=0) / 30.0)) \
         * np.sqrt(d['sample_size'].clip(lower=1)) \
         * (1.0 + d['numeric_grade'].fillna(0) / 3.0)

def wmean(values, weights):
    """Weighted mean, falling back to plain mean if weights are unusable."""
    w = weights.reindex(values.index).fillna(0)
    if w.sum() > 0 and values.notna().any():
        return np.average(values.fillna(values.mean()), weights=w)
    return values.mean()

## 3. Collapse to one row per candidate per race

Aggregate every poll for a candidate into a feature vector. Then add **race-relative** features (lead over best opponent, share of the polled field) — these matter more than raw % because elections are relative.

In [4]:
rows = []
for race_id, g in d.groupby('race_id'):
    for ck, gc in g.groupby('cand_key'):
        gc = gc.sort_values('end_date')
        last30 = gc[gc['days_to_elec'] <= 30]
        rows.append(dict(
            race_id=race_id, year=gc['year'].iloc[0], state=gc['state'].iloc[0],
            office=gc['office'].iloc[0], cand_key=ck, candidate=gc['candidate'].iloc[0],
            party=gc['party_std'].iloc[0], won=int(gc['won'].iloc[0]),
            poll_avg=gc['pct'].mean(),
            poll_wavg=wmean(gc['pct'], gc['w']),
            poll_last=gc['pct'].iloc[-1],
            poll_last30=(last30['pct'].mean() if len(last30) else gc['pct'].mean()),
            poll_std=gc['pct'].std(),
            n_polls=len(gc),
            avg_grade=gc['numeric_grade'].mean(),
            avg_pollscore=gc['pollscore'].mean(),
            avg_sample=gc['sample_size'].mean(),
            min_days=gc['days_to_elec'].min(),
        ))
cand = pd.DataFrame(rows)

# race-relative features
cand['field_best'] = cand.groupby('race_id')['poll_wavg'].transform(
    lambda s: s.nlargest(2).min() if len(s) > 1 else s.max())
cand['poll_lead']  = cand['poll_wavg'] - cand['field_best']
cand['poll_share'] = cand['poll_wavg'] / cand.groupby('race_id')['poll_wavg'].transform('sum')
cand['n_cands']    = cand.groupby('race_id')['cand_key'].transform('count')
cand['is_dem']     = (cand['party'] == 'DEM').astype(int)
cand['is_rep']     = (cand['party'] == 'REP').astype(int)
cand['is_senate']  = (cand['office'] == 'Senate').astype(int)
cand['is_gov']     = (cand['office'] == 'Governor').astype(int)

print('candidate-races:', len(cand), '| win rate:', round(cand['won'].mean(), 3))
cand.head()

candidate-races: 1857 | win rate: 0.37


,race_id,year,state,office,cand_key,candidate,party,won,poll_avg,poll_wavg,poll_last,poll_last30,poll_std,n_polls,avg_grade,avg_pollscore,avg_sample,min_days,field_best,poll_lead,poll_share,n_cands,is_dem,is_rep,is_senate,is_gov
0,2018_AK_Governor,2018,AK,Governor,begich m,Mark Begich,DEM,0,35.552941,35.388797,42.3,36.04,10.751228,17,1.623529,0.105882,574.588235,8,35.388797,0.000000,0.332126,4,1,0,0,1
1,2018_AK_Governor,2018,AK,Governor,dunleavy m,Mike Dunleavy,REP,1,42.918182,44.673941,42.5,45.78,5.671284,22,1.568182,0.140909,596.954545,8,35.388797,9.285143,0.419268,4,0,1,0,1
2,2018_AK_Governor,2018,AK,Governor,toien b,Billy Toien,OTH,0,3.300000,3.300000,3.3,3.30,NaN,1,1.900000,-0.100000,500.000000,8,35.388797,-32.088797,0.030971,4,0,0,0,1
3,2018_AK_Governor,2018,AK,Governor,walker b,Bill Walker,OTH,0,29.112500,23.189473,7.7,15.90,13.140567,16,1.556250,0.143750,604.250000,8,35.388797,-12.199325,0.217635,4,0,0,0,1
4,2018_AK_House,2018,AK,House,galvin a,Alyse S. Galvin,DEM,0,43.957143,45.252545,49.0,45.80,4.212623,7,1.628571,-0.028571,526.428571,8,45.252545,0.000000,0.482868,2,1,0,0,0


## 4. Train / test split by year

Train on 2018/2020/2022, test on 2024. **Never** include `vote_pct` / `race_winning_pct` — those are outcomes.

In [5]:
FEATURES = [
    'poll_avg','poll_wavg','poll_last','poll_last30','poll_std','n_polls',
    'avg_grade','avg_pollscore','avg_sample','min_days',
    'poll_lead','poll_share','n_cands',
    'is_dem','is_rep','is_senate','is_gov',
]

train = cand[cand['year'] <= 2022]
test  = cand[cand['year'] == 2024]
X_train, y_train = train[FEATURES], train['won']
X_test,  y_test  = test[FEATURES],  test['won']
print('train:', X_train.shape, '| test:', X_test.shape)
print('train win rate:', round(y_train.mean(), 3), '| test win rate:', round(y_test.mean(), 3))

train: (1514, 17) | test: (343, 17)
train win rate: 0.367 | test win rate: 0.388


## 5. Train XGBoost

Shallow trees + low learning rate + subsampling to avoid overfitting on a small dataset (~1.5k training rows).

In [6]:
model = xgb.XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    eval_metric='logloss', random_state=42,
)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred  = (proba > 0.5).astype(int)

## 6. Evaluate (candidate level)

In [7]:
print('=== candidate-level (test = 2024) ===')
print('AUC      ', round(roc_auc_score(y_test, proba), 3))
print('LogLoss  ', round(log_loss(y_test, proba), 3))
print('Brier    ', round(brier_score_loss(y_test, proba), 3), ' (lower = better calibrated)')
print('Accuracy ', round(accuracy_score(y_test, pred), 3))
print('\nConfusion matrix [rows=true, cols=pred]:')
print(confusion_matrix(y_test, pred))

=== candidate-level (test = 2024) ===
AUC       0.942
LogLoss   0.3
Brier     0.095  (lower = better calibrated)
Accuracy  0.863

Confusion matrix [rows=true, cols=pred]:
[[190  20]
 [ 27 106]]


## 7. Evaluate (race level) + baseline

Real question: per race, did we pick the right winner? Compare the model (highest predicted prob) to the naive baseline (highest polling average).

In [8]:
te = test.copy()
te['proba'] = proba

model_pick = te.loc[te.groupby('race_id')['proba'].idxmax()]
base_pick  = te.loc[te.groupby('race_id')['poll_wavg'].idxmax()]

print('races in test:', te['race_id'].nunique())
print('model  winner accuracy:', round(model_pick['won'].mean(), 3))
print('baseline (top poll)   :', round(base_pick['won'].mean(), 3))
print('\nRaces the model got WRONG:')
wrong = model_pick[model_pick['won'] == 0]
wrong[['year','state','office','candidate','party','poll_wavg','poll_lead','proba']]

races in test: 78
model  winner accuracy: 0.949
baseline (top poll)   : 0.936

Races the model got WRONG:


,year,state,office,candidate,party,poll_wavg,poll_lead,proba
1564,2024,CO,House,Adam Frisch,DEM,48.017227,2.280718,0.979075
1609,2024,MD,House,Neil C. Parrott,REP,40.644147,0.081089,0.463506
1770,2024,PA,Senate,Robert P. Casey Jr.,DEM,48.531721,4.444995,0.950234
1836,2024,WA,House,Joe Kent,REP,45.172090,0.214533,0.494278


## 8. Feature importance

In [9]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
ax = imp.plot.barh(figsize=(7, 5), title='XGBoost feature importance')
ax.set_xlabel('gain importance')
plt.tight_layout(); plt.show()
imp.sort_values(ascending=False).round(3)

C:\Users\pjmer\AppData\Local\Temp\ipykernel_21716\3632646110.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


poll_wavg        0.296
poll_avg         0.194
poll_last30      0.109
poll_last        0.044
poll_lead        0.040
avg_pollscore    0.035
is_rep           0.033
is_dem           0.031
poll_share       0.030
n_cands          0.030
min_days         0.029
avg_sample       0.026
avg_grade        0.024
poll_std         0.023
n_polls          0.021
is_gov           0.019
is_senate        0.017
dtype: float32

## 9. Calibration curve

Are predicted probabilities honest? A well-calibrated model: candidates given ~70% win ~70% of the time.

In [10]:
frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=8, strategy='quantile')
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], '--', color='gray', label='perfect')
plt.plot(mean_pred, frac_pos, 'o-', label='model')
plt.xlabel('predicted win probability'); plt.ylabel('actual win frequency')
plt.title('Calibration (2024 test)'); plt.legend(); plt.tight_layout(); plt.show()

C:\Users\pjmer\AppData\Local\Temp\ipykernel_21716\4115570372.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('Calibration (2024 test)'); plt.legend(); plt.tight_layout(); plt.show()


## Notes / next steps

- **Polls dominate** — the weighted average and lead carry most of the signal. Pollster-quality and recency features add calibration, not headline accuracy.
- **Small test set** (~78 races) → treat single-number differences cautiously. For a sturdier estimate, do leave-one-cycle-out CV (train on the other 3 cycles, test each in turn) and average.
- **Ideas to add:** incumbency (in the results/`races.csv` file), national environment (generic ballot), state partisan lean (prior-cycle margin), polls-vs-fundamentals blend, pollster house-effect adjustment.
- **Don't** add `vote_pct` / `race_winning_pct` as features — they're the answer.
- To predict a *live* race, run `build_dataset.ipynb` for the current cycle and score with this model (note: current-cycle rows have `has_result == 0`, no label yet).